# Weather agent

### Generate API key

https://openweathermap.org/

### 1. Basic Weather agent

In [1]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
import requests

# LLM
llm = ChatOllama(model="llama3.2")

# API KEY

OPENWEATHER_API_KEY = ''

In [2]:

# TOOL
@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a given city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    # Error handling
    if data.get("cod") != 200:
        return f"Error fetching weather: {data}"

    description = data["weather"][0]["description"]
    temperature = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    return (
        f"Weather in {city}:\n"
        f"Condition: {description}\n"
        f"Temperature: {temperature}°C\n"
        f"Humidity: {humidity}%"
    )


In [3]:
# CREATE AGENT

agent = create_agent(
    model=llm,
    tools=[get_weather]
)

In [4]:
# USER QUERY

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hyderabad?"
            }
        ]
    }
)


In [5]:

print(response)

{'messages': [HumanMessage(content='What is the weather in Hyderabad?', additional_kwargs={}, response_metadata={}, id='d493cb73-dc84-4f99-b74d-cca751ab5c22'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-06-16T04:49:30.5071795Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13436331800, 'load_duration': 8334128800, 'prompt_eval_count': 157, 'prompt_eval_duration': 3379709000, 'eval_count': 18, 'eval_duration': 1705078000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019ecec3-2907-7de1-bbcc-6281980d13a9-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': '187fcfe0-6fbd-498e-bf30-0afb42ab7b38', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 18, 'total_tokens': 175}), ToolMessage(content='Weather in Hyderabad:\nCondition: overcast clouds\nTemperature: 32.58°C\nHumidity: 41%', name='get_weather', id='269a

In [6]:

for i, msg in enumerate(response["messages"]):
    print(f"\nINDEX: {i}")
    print(msg)
    


INDEX: 0
content='What is the weather in Hyderabad?' additional_kwargs={} response_metadata={} id='d493cb73-dc84-4f99-b74d-cca751ab5c22'

INDEX: 1
content='' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-06-16T04:49:30.5071795Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13436331800, 'load_duration': 8334128800, 'prompt_eval_count': 157, 'prompt_eval_duration': 3379709000, 'eval_count': 18, 'eval_duration': 1705078000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--019ecec3-2907-7de1-bbcc-6281980d13a9-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': '187fcfe0-6fbd-498e-bf30-0afb42ab7b38', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 157, 'output_tokens': 18, 'total_tokens': 175}

INDEX: 2
content='Weather in Hyderabad:\nCondition: overcast clouds\nTemperature: 32.58°C\nHumidity: 41%' name='get_weather' id='269a46e1-cf96-4ed9-9e83-3aa5fd9f96cb' 

In [7]:

print(response["messages"][-1].content)

The current temperature in Hyderabad is around 32.58°C. It's a relatively warm day with an average humidity level of 41%. The weather condition is mostly overcast, indicating minimal sunshine today. Would you like to know the forecast for tomorrow or something else?


In [8]:

response["messages"][-2].content

'Weather in Hyderabad:\nCondition: overcast clouds\nTemperature: 32.58°C\nHumidity: 41%'

In [9]:

for i, msg in enumerate(response["messages"]):
    print(i, type(msg).__name__)
    print(msg.content)


0 HumanMessage
What is the weather in Hyderabad?
1 AIMessage

2 ToolMessage
Weather in Hyderabad:
Condition: overcast clouds
Temperature: 32.58°C
Humidity: 41%
3 AIMessage
The current temperature in Hyderabad is around 32.58°C. It's a relatively warm day with an average humidity level of 41%. The weather condition is mostly overcast, indicating minimal sunshine today. Would you like to know the forecast for tomorrow or something else?


### messages list:

| Index | Message Type               |
| ----- | -------------------------- |
| 0     | HumanMessage               |
| 1     | AIMessage (tool call)      |
| 2     | ToolMessage (weather data) |
| 3     | Final AIMessage            |


| cod | Meaning           |
| --- | ----------------- |
| 200 | Success           |
| 401 | Invalid API key   |
| 404 | City not found    |
| 429 | Too many requests |
| 500 | Server error      |


### 2. Multi-Tool AI Agent

In [10]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
import requests

In [11]:

# LLM
llm = ChatOllama(model="mistral:latest")

In [12]:

# WEATHER TOOL
@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    if data.get("cod") != 200:
        return "Weather data not available."

    description = data["weather"][0]["description"]
    temperature = data["main"]["temp"]

    return f"{city}: {description}, {temperature}°C"


# CALCULATOR TOOL
@tool
def calculator(expression: str) -> str:
    """
    Perform mathematical calculations.
    """

    try:
        result = eval(expression)
        return f"Result: {result}"

    except Exception as e:
        return f"Error: {str(e)}"


In [13]:
# CREATE AGENT
agent = create_agent(
    model=llm,
    tools=[get_weather, calculator]
)

# USER QUERY
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is weather in Hyderabad and calculate 25*8?"
        }
    ]
})


In [14]:
# FINAL RESPONSE

print(response["messages"][-1].content)

 The weather in Hyderabad is overcast clouds with a temperature of 32.58 degrees Celsius. The result of the calculation 25*8 is 200.


### 3 Conversational Memory Agent

In [15]:
from langchain_ollama import ChatOllama

from langchain.tools import tool

from langchain.agents import create_agent

In [16]:

# LLM
llm = ChatOllama(model="mistral:latest")

In [17]:
import requests

In [18]:

# WEATHER TOOL
@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    if data.get("cod") != 200:
        return "Weather data not available."

    description = data["weather"][0]["description"]
    temperature = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    return f"""
Weather in {city}
Condition: {description}
Temperature: {temperature}°C
Humidity: {humidity}%
"""


# CALCULATOR TOOL
@tool
def calculator(expression: str) -> str:
    """
    Perform mathematical calculations.
    """

    try:
        result = eval(expression)
        return f"Result: {result}"

    except Exception as e:
        return f"Error: {str(e)}"


In [19]:
# CREATE AGENT
agent = create_agent(
    model=llm,
    tools=[get_weather, calculator]
)


In [20]:
# MEMORY
chat_history = []

# CHAT LOOP
while True:

    user_input = input("\nYou: ")

    # EXIT CONDITION
    if user_input.lower() == "exit":
        break

    # ADD USER MESSAGE TO MEMORY
    chat_history.append({
        "role": "user",
        "content": user_input
    })

    # AGENT RESPONSE
    response = agent.invoke({
        "messages": chat_history
    })

    # FINAL AI MESSAGE
    ai_message = response["messages"][-1].content

    print("\nAI:", ai_message)

    # STORE AI RESPONSE IN MEMORY
    chat_history.append({
        "role": "assistant",
        "content": ai_message
    })


You:  what is the weather in Vizag?



AI:  The weather in Vizag is currently unavailable. Please check again later.



You:  what is the weather in Visakapatnam?4



AI:  The weather in Visakapatnam is currently unavailable. Please check again later.



You:  what is the weather in Bengaluru



AI:  The weather in Bengaluru is currently broken clouds, with a temperature of 27.79°C and humidity of 66%.



You:  exit


In [21]:
chat_history

[{'role': 'user', 'content': 'what is the weather in Vizag?'},
 {'role': 'assistant',
  'content': ' The weather in Vizag is currently unavailable. Please check again later.'},
 {'role': 'user', 'content': 'what is the weather in Visakapatnam?4'},
 {'role': 'assistant',
  'content': ' The weather in Visakapatnam is currently unavailable. Please check again later.'},
 {'role': 'user', 'content': 'what is the weather in Bengaluru'},
 {'role': 'assistant',
  'content': ' The weather in Bengaluru is currently broken clouds, with a temperature of 27.79°C and humidity of 66%.'}]

### 4 RAG AI AGENT (Knowledge + Weather)

In [22]:

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
import requests


In [23]:
# LLM

llm = ChatOllama(model="llama3.2")

In [24]:
# LOAD DOCUMENT

with open("data/weather_guide.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("TEXT:")
print(text)

TEXT:
Cyclone Safety Tips

1. Stay indoors
2. Keep emergency kits
3. Avoid flooded roads
4. Follow alerts


In [25]:
# SPLIT DOCUMENT

splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_text(text)

print("\nCHUNKS:")
print(chunks)

print("\nNUMBER OF CHUNKS:")
print(len(chunks))



CHUNKS:
['Cyclone Safety Tips\n\n1. Stay indoors\n2. Keep emergency kits\n3. Avoid flooded roads\n4. Follow alerts']

NUMBER OF CHUNKS:
1


In [27]:
# CREATE DOCUMENTS

documents = [Document(page_content=chunk) for chunk in chunks]
documents

[Document(metadata={}, page_content='Cyclone Safety Tips\n\n1. Stay indoors\n2. Keep emergency kits\n3. Avoid flooded roads\n4. Follow alerts')]

In [28]:
# EMBEDDINGS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Users\nvgra\anaconda3\annacond\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [29]:
# VECTOR STORE

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

In [30]:
# WEATHER TOOL

@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    if data.get("cod") != 200:
        return "Weather data not available."

    description = data["weather"][0]["description"]
    temperature = data["main"]["temp"]

    return f"{city}: {description}, {temperature}°C"


# RAG TOOL

@tool
def search_documents(query: str) -> str:
    """
    Search knowledge base documents.
    """

    docs = vectorstore.similarity_search(query, k=2)

    results = "\n".join([doc.page_content for doc in docs])

    return f"""
Knowledge Base Result:

{results}

Answer only using the above information.
Do not add extra knowledge.
"""


In [31]:

# CREATE AGENT

agent = create_agent(
    model=llm,
    tools=[get_weather, search_documents],
    system_prompt="""
You are a strict RAG assistant.

Rules:
1. Use ONLY provided tool information.
2. Do not add outside knowledge.
3. Do not hallucinate.
4. Keep answers concise.
"""
)


In [32]:
# MEMORY

chat_history = []

# CHAT LOOP

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        break

    chat_history.append({
        "role": "user",
        "content": user_input
    })

    response = agent.invoke({
        "messages": chat_history
    })

    ai_message = response["messages"][-1].content

    print("\nAI:", ai_message)

    chat_history.append({
        "role": "assistant",
        "content": ai_message
    })


You:  give me cyclone safety tips and Weather in hyderabad now



AI: {"name":"search_documents","parameters\":{\"query\":\"cyclone safety tips\"}}; {"name":"get_weather","parameters\":{\"city\":\"Hyderabad\"}}



You:  give me cyclone safety tips



AI: **Cyclone Safety Tips**

1. Stay indoors to ensure your safety during a cyclone.
2. Keep emergency kits with essential items such as food, water, first aid supplies, and a battery-powered radio.
3. Avoid traveling on flooded roads, as they can be slippery and hide hidden dangers.
4. Follow local alerts and instructions from authorities for the best course of action.

**Current Weather in Hyderabad:** Overcast clouds, 32.58°C



You:  exit


### 5 Agentic AI Reasoning

In [33]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
import requests


# LLM

llm = ChatOllama(model="llama3.2")

# WEATHER TOOL

@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    if data.get("cod") != 200:
        return "Weather data not available."

    description = data["weather"][0]["description"]

    temperature = data["main"]["temp"]

    humidity = data["main"]["humidity"]

    return f"""
City: {city}
Condition: {description}
Temperature: {temperature}°C
Humidity: {humidity}%
"""


# FITNESS TOOL

@tool
def jogging_advice(weather_report: str) -> str:
    """
    Suggest jogging advice based on weather.
    """

    report = weather_report.lower()

    if "rain" in report:
        return "Not ideal for jogging due to rain."

    elif "haze" in report:
        return "Air quality may be poor. Morning jogging is better."

    elif "clear" in report:
        return "Excellent weather for jogging."

    elif "hot" in report:
        return "Avoid afternoon jogging. Prefer early morning."

    else:
        return "Weather is moderate for jogging."


# CREATE AGENT

agent = create_agent(
    model=llm,
    tools=[get_weather, jogging_advice],
    system_prompt="""
You are a fitness and weather assistant.

Workflow:
1. First check weather.
2. Then analyze weather.
3. Then provide jogging advice.
4. Keep answers short and practical.
"""
)


# CHAT LOOP

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        break

    response = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_input
            }
        ]
    })

    print("\nAI:")
    print(response["messages"][-1].content)


You:  my city is hyderabad. Suggest jogging conditions for today



AI:
Considering the overcast conditions, it's a good idea to jog early in the morning or later in the evening when the sun isn't too strong. Wear light and breathable clothing to stay comfortable. Also, be mindful of the heat index with the temperature being 32.58°C and humidity at 41%.



You:  exit


### prompts

In [ ]:
Is it good to go for jogging today?
Can I run outside in this weather?
Suggest jogging conditions for today
Should I exercise outdoors now?
Is this weather safe for running?
Best time for jogging today?

### The agent workflow became:

User Query
   ↓
Need weather information
   ↓
Call weather tool
   ↓
Receive:
32°C + 43% humidity
   ↓
Reason:
"too hot in afternoon"
   ↓
Suggest:
morning/evening jogging
   ↓
Generate plan

### 6 Planner Agent

Step 1 → Check weather
Step 2 → Analyze temperature
Step 3 → Analyze humidity
Step 4 → Decide safe jogging time
Step 5 → Generate exercise plan
Step 6 → Add hydration advice

In [34]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
import requests


In [35]:
# LLM

llm = ChatOllama(model="llama3.2")


In [36]:
# WEATHER TOOL

@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a city.
    """

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    if data.get("cod") != 200:
        return "Weather data not available."

    description = data["weather"][0]["description"]

    temperature = data["main"]["temp"]

    humidity = data["main"]["humidity"]

    return f"""
City: {city}
Condition: {description}
Temperature: {temperature}
Humidity: {humidity}
"""


In [37]:
# PLANNER TOOL

@tool
def create_jogging_plan(weather_report: str) -> str:
    """
    Create jogging plan using weather conditions.
    """

    report = weather_report.lower()

    advice = []

    if "32" in report or "33" in report or "34" in report:
        advice.append("Avoid afternoon jogging")

    if "humidity" in report:
        advice.append("Carry water bottle")

    if "haze" in report:
        advice.append("Prefer indoor exercise if air quality worsens")

    advice.append("Best jogging time: 6 AM to 8 AM")

    advice.append("Jogging duration: 30 minutes")

    return "\n".join(advice)


In [38]:

# AGENT

agent = create_agent(
    model=llm,
    tools=[get_weather, create_jogging_plan],
    system_prompt="""
You are an autonomous fitness planning agent.

Workflow:
1. Check weather.
2. Analyze conditions carefully.
3. Create jogging strategy.
4. Give practical recommendations.
5. Explain reasoning briefly.
"""
)



In [ ]:

# LOOP

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        break

    response = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_input
            }
        ]
    })

    print("\nAI:")
    print(response["messages"][-1].content)


You:  safe jogging routine



AI:
Based on the weather conditions in New York, I recommend the following safe jogging routine:

**Jogging Strategy:**

1. Start with a 5-minute warm-up walk to get your blood flowing and muscles ready.
2. Begin with a gentle jog at a pace of 30 minutes per mile for the first 20 minutes.
3. After 20 minutes, increase the intensity by adding short bursts of faster jogging (about 10-15 seconds) followed by walking breaks (about 1-2 minutes).
4. Continue this pattern for the remaining 40 minutes of your jog.
5. Finish with a 5-minute cool-down walk to stretch and bring your heart rate back down.

**Practical Recommendations:**

* Wear comfortable and breathable clothing, including a moisture-wicking shirt and supportive socks.
* Choose jogging shoes that are designed for running on roads or sidewalks.
* Bring water and snacks (e.g., energy gels or bananas) to keep you hydrated and energized throughout your jog.
* Consider jogging with a buddy or using a fitness tracker to monitor your p

### Agent Is Doing

User Goal:
"safe jogging routine"

        ↓

Need weather context

        ↓

Call weather tool

        ↓

Analyze:
- haze
- temperature
- humidity

        ↓

Infer risks:
- heat exhaustion
- dehydration

        ↓

Generate plan:
- timing
- duration
- precautions

https://chatgpt.com/share/6a30b576-d64c-83ee-a586-8a503834aed3